# Thesis I: bounded Gaussians (Kaggle, unattended, ~6 h)

Fixes the accuracy gap found in the last run: the voxel grid is clipped to [0, 1] but the summed Gaussians were not.
- **Bounded density** f = 1 − exp(−Σ Gaussians), always in [0, 1).
- **Grid-initialised Gaussians**: start from the grid solution, then refine (a compact version of it).
- True 3D model and slice-wise on BBBC024, 2D on 10 cells, synthetic 3D.

**Settings:** Accelerator = **GPU T4 x2** (or P100), **Internet = On** → **Save Version → Save & Run All (Commit)**.
Output: **`thesis1_bounded_results.zip`** (Output tab).


In [ ]:
REPO_URL = "https://github.com/muhammadhussainqureshi/thesis-gs"   # change after transferring the repo
CONFIG = "thesis1_bounded"    # configs/<CONFIG>.json
QUICK = False               # True = few-minute smoke test with tiny settings

import os, subprocess, sys, glob, zipfile, shutil, torch
WORK = "/kaggle/working"
CODE = "/kaggle/working/thesis-gs"
DATA = f"{WORK}/data"
OUT = f"{WORK}/{CONFIG}"
print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "NONE - set Accelerator to GPU")
if os.path.isdir(CODE):
    shutil.rmtree(CODE)
subprocess.run(["git", "clone", "--depth", "1", REPO_URL, CODE], check=True)
assert os.path.isfile(f"{CODE}/configs/{CONFIG}.json"), "config not found - is the latest code pushed?"
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "lmdb", "tifffile"], check=True)
print(subprocess.run(["git", "-C", CODE, "log", "-1", "--oneline"], capture_output=True, text=True).stdout)


In [ ]:
# Real 3D cells: BBBC024 ground-truth masks (30 .ics volumes, 20 HL60 nuclei each, made with CytoPacq)
VOL = "/tmp/cytopacq"
os.makedirs(VOL, exist_ok=True)
name = "BBBC024_v1_c00_highSNR_foreground.zip"
zp = f"{VOL}/{name}"
try:
    if not os.path.exists(zp):
        subprocess.run(["wget", "-q", "-O", zp, f"https://data.broadinstitute.org/bbbc/BBBC024/{name}"])
    zipfile.ZipFile(zp).extractall(VOL)
    print(len(glob.glob(f"{VOL}/**/*.ics", recursive=True)), "BBBC024 volumes")
except Exception as e:
    print("BBBC024 download failed (BBBC024 runs will be skipped):", e)


In [ ]:
extra = ["--quick", "--transmitters", "8"] if QUICK else []
cmd = [sys.executable, f"{CODE}/bin/run_experiments.py", f"{CODE}/configs/{CONFIG}.json", "--out", OUT,
       "--data_dir", DATA, "--volumes", VOL] + extra
subprocess.run(cmd)        # failures are logged per run; the notebook keeps going


In [ ]:
import pandas as pd
from IPython.display import Image, display
pd.set_option("display.width", 200)
for f in sorted(glob.glob(f"{OUT}/**/summary.csv", recursive=True)):
    print("\n=====", os.path.relpath(os.path.dirname(f), OUT))
    print(pd.read_csv(f).to_string(index=False))
    png = os.path.join(os.path.dirname(f), "reconstructions.png")
    if os.path.exists(png):
        display(Image(filename=png))
for f in glob.glob(f"{OUT}/**/*.pt", recursive=True):   # keep the zip small
    os.remove(f)
os.makedirs(OUT, exist_ok=True)
shutil.make_archive(f"{WORK}/{CONFIG}_results", "zip", OUT)
print(f"\nDownload {CONFIG}_results.zip from the Output tab.")
